# 1. CombineClassifier

In [1]:
import time
import numpy as np
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from cobra.combine_classifier import CombineClassifier

X, y = make_classification(n_samples=10000, n_features=20, n_classes=2, random_state=42)
X_train, X_test = X[:8000], X[8000:]
y_train, y_test = y[:8000], y[8000:]

In [2]:
import numpy as np

def generate_estimators(n_total=30):
    estimators = []

    # Logistic Regression (8)
    for c in np.logspace(-2, 2, 8):
        estimators.append(("logistic_regression", {"C": c}))

    # Random Forest (8)
    for n in [10, 50, 100, 150, 200, 250, 300, 400]:
        estimators.append(("random_forest_classifier", {"n_estimators": n}))

    # SVC (7)
    for g in np.logspace(-3, 1, 7):
        estimators.append(("svc", {"gamma": g}))

    # KNN (7)
    for k in [3, 5, 7, 9, 11, 13, 15]:
        estimators.append(("k_neighbors_classifier", {"n_neighbors": k}))

    return estimators[:n_total]

In [3]:
# old
model1 = CombineClassifier()
start = time.time()
model1.fit(X_train, y_train)
fit_time = time.time() - start

start = time.time()
y_pred1 = model1.predict(X_test)
pred_time = time.time() - start

print(f"Original fit: {fit_time:.3f}s, predict: {pred_time:.3f}s, accuracy : {accuracy_score(y_test, y_pred1)}")

Original fit: 0.905s, predict: 0.597s, accuracy : 0.924


In [ ]:
# new
from cobra.combine_classifier_v2 import CombineClassifier as CombineClassifierV2
# Test original
model = CombineClassifierV2()
start = time.time()
model.fit(X_train, y_train)
fit_time = time.time() - start

start = time.time()
y_pred = model.predict(X_test)
pred_time = time.time() - start

print(f"Original fit: {fit_time:.3f}s, predict: {pred_time:.3f}s, accuracy : {accuracy_score(y_test, y_pred)}")

In [ ]:
estimators = generate_estimators(30)

In [ ]:
model = CombineClassifier(
    estimators=estimators
)
start = time.time()
model.fit(X_train, y_train)
fit_time = time.time() - start

start = time.time()
y_pred = model.predict(X_test)
pred_time = time.time() - start

print(f"Original fit: {fit_time:.3f}s, predict: {pred_time:.3f}s, accuracy : {accuracy_score(y_test, y_pred)}")

Original fit: 31.330s, predict: 2.434s, accuracy : 0.9135


In [ ]:
model = CombineClassifierV2(
    estimators=estimators
)
start = time.time()
model.fit(X_train, y_train)
fit_time = time.time() - start

start = time.time()
y_pred = model.predict(X_test)
pred_time = time.time() - start

print(f"Original fit: {fit_time:.3f}s, predict: {pred_time:.3f}s, accuracy : {accuracy_score(y_test, y_pred)}")

Original fit: 31.041s, predict: 2.418s, accuracy : 0.914


In [ ]:
from cobra.combine_classifier_v2 import CombineClassifierFast

model = CombineClassifierFast(
    estimators=estimators,
    use_faiss=True,
    faiss_k=10
)
start = time.time()
model.fit(X_train, y_train)
fit_time = time.time() - start

start = time.time()
y_pred = model.predict(X_test)
pred_time = time.time() - start

print(f"Original fit: {fit_time:.3f}s, predict: {pred_time:.3f}s, accuracy : {accuracy_score(y_test, y_pred)}")

Original fit: 31.193s, predict: 2.051s, accuracy : 0.9175


In [ ]:
from cobra.core.estimators.base import EstimatorFactory

estimators = generate_estimators(30)
models = {}

models["combine_classifier"] = CombineClassifierFast(
    estimators=estimators,
    use_faiss=True,
    faiss_k=10
)

for index, (name, params) in enumerate(estimators):
    model = EstimatorFactory.create(name, **(params or {}))
    models[f"{index + 1}{name}"] = model

results = []

for name, model in models.items():
    # ---- train ----
    start = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - start
    # ---- predict ----
    start = time.time()
    y_pred = model.predict(X_test)
    pred_time = time.time() - start
    # ---- accuracy ----
    acc = accuracy_score(y_test, y_pred)
    results.append({
        "model": name,
        "accuracy": acc,
        "fit_time": fit_time,
        "pred_time": pred_time
    })
    print(f"{name:25s} | acc={acc:.4f} | fit={fit_time:.3f}s | pred={pred_time:.3f}s")


/Users/ougi/Documents/Project/kfc-procedure/.venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


combine_classifier        | acc=0.9180 | fit=8.345s | pred=1.623s
1logistic_regression      | acc=0.8930 | fit=0.002s | pred=0.000s
2logistic_regression      | acc=0.8935 | fit=0.002s | pred=0.000s
3logistic_regression      | acc=0.8930 | fit=0.001s | pred=0.000s
4logistic_regression      | acc=0.8920 | fit=0.001s | pred=0.000s
5logistic_regression      | acc=0.8920 | fit=0.001s | pred=0.000s
6logistic_regression      | acc=0.8920 | fit=0.001s | pred=0.000s
7logistic_regression      | acc=0.8920 | fit=0.001s | pred=0.000s
8logistic_regression      | acc=0.8920 | fit=0.001s | pred=0.000s
9random_forest_classifier | acc=0.9305 | fit=0.130s | pred=0.001s
10random_forest_classifier | acc=0.9375 | fit=0.711s | pred=0.007s
11random_forest_classifier | acc=0.9365 | fit=1.429s | pred=0.013s
12random_forest_classifier | acc=0.9375 | fit=2.098s | pred=0.019s
13random_forest_classifier | acc=0.9385 | fit=2.724s | pred=0.026s
14random_forest_classifier | acc=0.9370 | fit=3.378s | pred=0.031s
15ran

In [ ]:
print("\n================ RANKING (by accuracy) ================\n")
results = sorted(results, key=lambda x: x["accuracy"], reverse=True)
for idx, r in enumerate(results):
    print(f"Rank {idx + 1} : {r['model']:25s} → acc={r['accuracy']:.4f}")


================ RANKING (by accuracy) ================

Rank 1 : 16random_forest_classifier → acc=0.9395
Rank 2 : 13random_forest_classifier → acc=0.9385
Rank 3 : 10random_forest_classifier → acc=0.9375
Rank 4 : 12random_forest_classifier → acc=0.9375
Rank 5 : 14random_forest_classifier → acc=0.9370
Rank 6 : 11random_forest_classifier → acc=0.9365
Rank 7 : 15random_forest_classifier → acc=0.9355
Rank 8 : 9random_forest_classifier → acc=0.9305
Rank 9 : 20svc                     → acc=0.9230
Rank 10 : 19svc                     → acc=0.9225
Rank 11 : combine_classifier        → acc=0.9180
Rank 12 : 18svc                     → acc=0.9115
Rank 13 : 17svc                     → acc=0.8985
Rank 14 : 28k_neighbors_classifier  → acc=0.8980
Rank 15 : 30k_neighbors_classifier  → acc=0.8970
Rank 16 : 29k_neighbors_classifier  → acc=0.8960
Rank 17 : 2logistic_regression      → acc=0.8935
Rank 18 : 1logistic_regression      → acc=0.8930
Rank 19 : 3logistic_regression      → acc=0.8930
Rank 20 : 4lo

# 2. GradientCobra

In [1]:
import time
from sklearn.metrics import mean_squared_error
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from cobra.core.estimators import EstimatorFactory

X, y = make_regression(n_samples=10000, n_features=2, noise=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [ ]:
from cobra.gradientcobra import GradientCOBRA

model = GradientCOBRA()
start = time.time()
model.fit(X_train, y_train)
fit_time = time.time() - start

start = time.time()
y_pred = model.predict(X_test)
pred_time = time.time() - start
print(f"Original fit: {fit_time:.3f}s, predict: {pred_time:.3f}s, mse: {mean_squared_error(y_test, y_pred)}")

GD iter: 99 | grad norm: 0.0120: 100%|██████████| 100/100 [00:42<00:00,  2.33it/s]


Original fit: 43.576s, predict: 0.316s, mse: 9764.945426733357


In [ ]:
from gradientcobra.gradientcobra import GradientCOBRA as OldGradientCOBRA

model = OldGradientCOBRA()
start = time.time()
model.fit(X_train, y_train)
fit_time = time.time() - start
start = time.time()
y_pred = model.predict(X_test)
pred_time = time.time() - start
print(f"Original fit: {fit_time:.3f}s, predict: {pred_time:.3f}s, mse: {mean_squared_error(y_test, y_pred)}")


* GD progress: iter: 300 / bw: 7.204 / grad: -0.75 / stop criter: [2.346] : 100%|██████████| 300/300 [00:39<00:00,  7.66it/s] 


Original fit: 40.449s, predict: 0.473s, mse: 4.120982121831274


In [ ]:
linear = EstimatorFactory.create("linear_regression")
linear.fit(X_train, y_train)
y_pred = linear.predict(X_test)
print(f"mse: {mean_squared_error(y_test, y_pred)}")

mse: 0.9957692375623143


In [ ]:
import numpy as np
from cobra.gradientcobra_v2 import GradientCOBRA as GradientCOBRAV2

model = GradientCOBRAV2(opt_method='grid')
start = time.time()
model.fit(X_train, y_train)
fit_time = time.time() - start
start = time.time()
y_pred = model.predict(X_test)
pred_time = time.time() - start
print(f"Original fit: {fit_time:.3f}s, predict: {pred_time:.3f}s, mse: {mean_squared_error(y_test, y_pred)}")


Adam: 100%|██████████| 100/100 [00:14<00:00,  6.84it/s, loss=5.173546, x=0.0826]  


Original fit: 18.147s, predict: 1.431s, mse: 5995.661786318363
